<h1 style="text-align:center"><b>[Step 2]</b></h1>
<h2 style="text-align:center"><b>Labeling Process</b></h2>
<p style="text-align:center">Combining LLMs with Gathered Surveys of Real Human Perspectives</p>

In [1]:
import pandas as pd
import numpy as np
from transformers import pipeline
import re
import time
import os
import torch
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"

c:\Users\justi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\justi\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
pip install openpyxl


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import numpy as np

gold = pd.read_excel("Clean-Thesis_Labeled_Gold.xlsx")
llm  = pd.read_csv("indo_fake_review.csv")

# Samakan tipe id
gold["id"] = pd.to_numeric(gold["id"], errors="coerce").astype("Int64")
llm["id"]  = pd.to_numeric(llm["id"],  errors="coerce").astype("Int64")

# Merge: ambil semua row dari LLM, tempel label + teks gold jika ada
df = llm.merge(
    gold[["id", "ulasan_cleaned", "Label"]],
    on="id",
    how="left"
)

# Pilih teks untuk training (prioritas original_text)
df["text_final"] = df["original_text"].fillna(df["ulasan_cleaned"])

# Label final (prioritas Gold)
df["label_final"] = df["Label"].fillna(df["decision"])

# Tambahan: sumber teks & label (ini yang bikin "hybrid" lebih jelas)
df["text_source"] = np.where(df["original_text"].notna(), "llm_original_text", "gold_ulasan_cleaned")
df["label_source"] = np.where(df["Label"].notna(), "gold", "llm")

# (Opsional) biner
df["label_bin"] = df["label_final"].map({"Fake": 1, "Real": 0})

# Quick checks
print("gold rows:", len(gold), "llm rows:", len(llm), "merged rows:", len(df))
print("id null:", df["id"].isna().sum())
print("text_final null:", df["text_final"].isna().sum())
print(df["label_source"].value_counts(dropna=False))

df.to_csv("dataset_final.csv", index=False)


gold rows: 1965 llm rows: 7875 merged rows: 7875
id null: 0
text_final null: 0
label_source
llm     5910
gold    1965
Name: count, dtype: int64


In [4]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score

# ====== load ======
df = pd.read_csv("dataset_final.csv")

# rapikan label (hindari spasi/kapital beda)
for col in ["decision", "Label"]:
    df[col] = df[col].astype(str).str.strip().str.capitalize()

# mask gold: yang punya Label (human)
mask_gold = df["Label"].notna() & (df["Label"].astype(str).str.lower() != "nan")
gold_df = df[mask_gold].copy()

# mapping biner
map_bin = {"Real": 0, "Fake": 1}
gold_df["y_true"] = gold_df["Label"].map(map_bin)
gold_df["y_pred"] = gold_df["decision"].map(map_bin)

# buang row yang tidak bisa dimap (jaga-jaga)
gold_df = gold_df[gold_df["y_true"].notna() & gold_df["y_pred"].notna()].copy()

# ====== METRIK ======
acc = accuracy_score(gold_df["y_true"], gold_df["y_pred"])
f1  = f1_score(gold_df["y_true"], gold_df["y_pred"])  # Fake sebagai positive class (1)
cm  = confusion_matrix(gold_df["y_true"], gold_df["y_pred"], labels=[0,1])

report = classification_report(
    gold_df["y_true"], gold_df["y_pred"],
    target_names=["Real(0)", "Fake(1)"],
    output_dict=True
)

metrics = {
    "gold_rows_evaluated": len(gold_df),
    "accuracy": acc,
    "f1_fake_pos": f1,
    "macro_f1": report["macro avg"]["f1-score"],
    "weighted_f1": report["weighted avg"]["f1-score"],
    "cm_real_real": int(cm[0,0]),
    "cm_real_fake": int(cm[0,1]),
    "cm_fake_real": int(cm[1,0]),
    "cm_fake_fake": int(cm[1,1]),
}

pd.DataFrame([metrics]).to_csv("gold_llm_metrics.csv", index=False)

# ====== DAFTAR KASUS (1) disagreement pada gold ======
gold_disagree = gold_df[gold_df["y_true"] != gold_df["y_pred"]].copy()
cols = [c for c in ["id","original_text","text_final","ulasan_cleaned","decision","Label","confidence","suspicion_score"]
        if c in df.columns]
gold_disagree[cols].to_csv("gold_disagreements.csv", index=False)

# ====== DAFTAR KASUS (2) low confidence pada non-gold ======
df["confidence"] = pd.to_numeric(df.get("confidence", np.nan), errors="coerce")

mask_non_gold = ~mask_gold
thr_low = 0.85
to_review_low = df[mask_non_gold & df["confidence"].notna() & (df["confidence"] < thr_low)].copy()
to_review_low[cols].to_csv("to_review_low_confidence.csv", index=False)

# ====== DAFTAR KASUS (3) sampel high confidence pada non-gold ======
thr_high = 0.95
pool_high = df[mask_non_gold & df["confidence"].notna() & (df["confidence"] >= thr_high)].copy()

n_sample = min(200, max(30, int(0.02 * len(pool_high)))) if len(pool_high) > 0 else 0
to_review_high = pool_high.sample(n=n_sample, random_state=42) if n_sample > 0 else pool_high

to_review_high[cols].to_csv("to_review_high_confidence_sample.csv", index=False)

print("Saved:",
      "gold_llm_metrics.csv, gold_disagreements.csv, to_review_low_confidence.csv, to_review_high_confidence_sample.csv")
print("Counts:",
      "gold evaluated =", len(gold_df),
      "| gold disagreements =", len(gold_disagree),
      "| low conf (non-gold) =", len(to_review_low),
      "| high conf sample =", len(to_review_high))


PermissionError: [Errno 13] Permission denied: 'gold_llm_metrics.csv'

In [ ]:
import pandas as pd

df = pd.read_csv("dataset_final.csv")

# gold: ada Label
mask_gold = df["Label"].notna()

# pseudo-label aman: tidak ada Label, confidence tinggi
df["confidence"] = pd.to_numeric(df["confidence"], errors="coerce")
thr = 0.95
mask_pseudo = (~mask_gold) & (df["confidence"] >= thr)

gold_data   = df[mask_gold].copy()
pseudo_data = df[mask_pseudo].copy()  # ini hasil “confidence tinggi setelah QC”

# final dataset untuk training hybrid (kalau mau)
hybrid = pd.concat([gold_data, pseudo_data], ignore_index=True)

# X dan y
X = hybrid["text_final"]
y = hybrid["label_final"]   # atau label_bin kalau mau biner
